# SinhalaCheck — Module 1: Final Model Refit

**Project:** R26-IT-158 | Kaweeshwara P.D.S. (IT22331304)

Cross-validation already selected the configuration: **LaBSE (`setu4993/LaBSE`)**, 512 tokens,
3 epochs, lr 2e-5, macro-F1 **0.7534 ± 0.013** across 5 folds. This notebook only refits that
configuration once and saves it. Nothing is re-measured — about **8 minutes**.

### Getting the model out of Colab

Colab deletes `/content` when the runtime ends, which is how the last copy was lost. This notebook
offers three routes and you should take at least one:

1. **Hugging Face Hub** (recommended) — a permanent URL, and it solves the GitHub problem too:
   the model is 1.9GB and GitHub caps files at 100MB, so the repo can reference the Hub instead.
2. **Google Drive** — retry the mount; it failed last time with a credentials error.
3. **Direct download** — always available, but ~1.9GB through the browser.

### Before you run
- Runtime → Change runtime type → **T4 GPU**
- Have `Corpus.xlsx` ready.

## 1 — Setup

In [ ]:
!pip install -q transformers torch scikit-learn pandas openpyxl huggingface_hub
import os, re, random, warnings, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

# ---- the configuration cross-validation selected
CHECKPOINT    = "setu4993/LaBSE"
MAX_LENGTH    = 512
EPOCHS        = 3
LEARNING_RATE = 2e-5
BATCH_SIZE    = 8
print(f"Refitting {CHECKPOINT} @ {MAX_LENGTH} tokens, {EPOCHS} epochs, lr={LEARNING_RATE}")

In [ ]:
from google.colab import files
if not os.path.exists("Corpus.xlsx"):
    print("Upload Corpus.xlsx:")
    up = files.upload(); src = list(up.keys())[0]
    if src != "Corpus.xlsx": os.rename(src, "Corpus.xlsx")
print("Corpus.xlsx present.")

## 2 — Data

Identical preparation to the validated run: the Excel boolean-`FALSE` repair, binary labels with
`CREDIBLE = 1`, and the same stratified 80/20 split with `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

def normalise_type(v):
    if isinstance(v, bool): return "FALSE" if v is False else "TRUE"
    return str(v).strip().upper()

df = pd.read_excel("Corpus.xlsx")
data = pd.DataFrame({"text": df["content"].astype(str),
                     "type": df["type"].apply(normalise_type)})
data = data[data["text"].str.strip().str.len() > 0].reset_index(drop=True)
data["label"] = (data["type"] == "CREDIBLE").astype(int)

y = data["label"].values
texts = data["text"].tolist()
idx_tr, idx_te = train_test_split(np.arange(len(data)), test_size=0.2,
                                  random_state=SEED, stratify=y)
print(f"{len(data)} documents | train {len(idx_tr)} | test {len(idx_te)}")
print(data["type"].value_counts().to_string())

## 3 — Train

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report

class DS(Dataset):
    def __init__(self, txt, lab, tok):
        self.enc = tok(list(txt), truncation=True, padding="max_length",
                       max_length=MAX_LENGTH, return_tensors="pt")
        self.lab = torch.tensor(np.asarray(lab), dtype=torch.long)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}; d["labels"] = self.lab[i]; return d

set_seed()
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=2,
    id2label={0: "NOT_CREDIBLE", 1: "CREDIBLE"},
    label2id={"NOT_CREDIBLE": 0, "CREDIBLE": 1}).to(device)

tr = DataLoader(DS([texts[i] for i in idx_tr], y[idx_tr], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
te = DataLoader(DS([texts[i] for i in idx_te], y[idx_te], tokenizer), batch_size=BATCH_SIZE)

opt = AdamW(model.parameters(), lr=LEARNING_RATE)
total = len(tr) * EPOCHS
sch = get_linear_schedule_with_warmup(opt, total // 10, total)
try:    scaler = torch.amp.GradScaler("cuda", enabled=(device.type=="cuda"))
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

for ep in range(EPOCHS):
    model.train(); run = 0.0
    for step, b in enumerate(tr):
        b = {k: v.to(device) for k, v in b.items()}
        opt.zero_grad()
        try:    ctx = torch.amp.autocast("cuda", enabled=(device.type=="cuda"))
        except Exception: ctx = torch.cuda.amp.autocast(enabled=(device.type=="cuda"))
        with ctx: loss = model(**b).loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        run += loss.item()
        if step % 50 == 0: print(f"  epoch {ep+1}/{EPOCHS}  step {step}/{len(tr)}  loss {loss.item():.4f}")
    print(f"-> epoch {ep+1} mean loss {run/len(tr):.4f}")

model.eval(); preds = []
with torch.no_grad():
    for b in te:
        b = {k: v.to(device) for k, v in b.items()}
        preds.extend(model(input_ids=b["input_ids"],
                           attention_mask=b["attention_mask"]).logits.argmax(-1).cpu().numpy())
preds = np.asarray(preds); yte = y[idx_te]

print("\n" + "="*70)
print(f"Held-out accuracy {accuracy_score(yte,preds):.4f} | macro-F1 {f1_score(yte,preds,average='macro'):.4f}")
print("="*70)
print(classification_report(yte, preds, target_names=["NOT CREDIBLE","CREDIBLE"], zero_division=0))
print("Cross-validated reference: macro-F1 0.7534 +/- 0.013")
print("A single split will differ a little from the CV mean - that is expected, not a problem.")

## 4 — Save locally

Saved in the exact layout `app.py` expects, so pointing it at this folder needs no code change.
`id2label` is written into the config, which removes any ambiguity about which index means
CREDIBLE.

In [ ]:
OUT = "/content/SinhalaCheck_model_final"
model.save_pretrained(OUT); tokenizer.save_pretrained(OUT)
print(f"Saved -> {OUT}")
!du -sh $OUT
!ls -la $OUT

## 5 — Route A: push to Hugging Face Hub (recommended)

This gives the model a permanent home, survives the runtime dying, and lets the GitHub repo
reference it by name instead of trying to commit 1.9GB.

**One-time setup:** create a free account at huggingface.co, then Settings → Access Tokens →
New token → type **Write**. Paste it when prompted below.

Set `HF_USERNAME` to your Hugging Face username first. Skip this cell if you'd rather not.

In [ ]:
HF_USERNAME = ""          # <-- your huggingface.co username, e.g. "kaweeshwara"
REPO_NAME   = "sinhalacheck-module1-labse"

if not HF_USERNAME:
    print("HF_USERNAME is blank - skipping. Use section 6 or 7 instead.")
else:
    from huggingface_hub import login, create_repo
    login()                                    # paste your write token at the prompt
    repo_id = f"{HF_USERNAME}/{REPO_NAME}"
    create_repo(repo_id, exist_ok=True, private=False)
    model.push_to_hub(repo_id); tokenizer.push_to_hub(repo_id)
    print(f"\nPushed -> https://huggingface.co/{repo_id}")
    print("\nFrom now on, anywhere that loads the model can simply use:")
    print(f'  AutoModelForSequenceClassification.from_pretrained("{repo_id}")')

## 6 — Route B: Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r $OUT /content/drive/MyDrive/SinhalaCheck_model_final
    print("Copied to Drive -> SinhalaCheck_model_final")
except Exception as e:
    print(f"Drive mount failed again ({type(e).__name__}).")
    print("This is usually third-party cookies being blocked. Either allow them for")
    print("google.com in your browser settings, or just use route A or C.")

## 7 — Route C: direct download

In [ ]:
!cd /content && zip -qr sinhalacheck_model_final.zip SinhalaCheck_model_final
!du -sh /content/sinhalacheck_model_final.zip
print("\nDownload it from the folder icon in the left sidebar (3-dot menu -> Download).")
print("It is roughly 1.9GB, so expect a slow download - route A is quicker if available.")